JPMorgan Chase Quantitative Research - Task 2 (Version Complète)
Advanced Contract Pricing Model for Natural Gas Storage

This script calculates the value of a natural gas storage contract
with multiple injection/withdrawal dates and operational constraints.

Key Features:
- Multiple injection dates
- Multiple withdrawal dates
- Injection/withdrawal rate constraints
- Maximum storage volume constraint
- Storage cost calculation

In [1]:
import pandas as pd
import numpy as np
from scipy import interpolate
from scipy.optimize import curve_fit
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

In [2]:
# ============================================================================
# 1. FUNCTION estimate_price
# ============================================================================

# Load historical data
df = pd.read_csv('Nat_Gas2.csv')
df['Dates'] = pd.to_datetime(df['Dates'], format='%m/%d/%y')
df['Prices'] = df['Prices'].astype(float)
df = df.sort_values('Dates').reset_index(drop=True)

# Seasonality analysis
df['Month'] = df['Dates'].dt.month
monthly_avg = df.groupby('Month')['Prices'].mean()


def estimate_price(target_date):
    """Estimate the price of natural gas for a given date."""
    if isinstance(target_date, str):
        target_date = pd.to_datetime(target_date)
    
    dates = df['Dates'].values
    prices = df['Prices'].values
    dates_numeric = np.array([(d - dates[0]).days for d in pd.to_datetime(dates)])
    target_numeric = (target_date - pd.to_datetime(dates[0])).days
    
    min_date = pd.to_datetime(dates[0])
    max_date = pd.to_datetime(dates[-1])
    
    if min_date <= target_date <= max_date:
        interpolator = interpolate.interp1d(dates_numeric, prices, 
                                            kind='cubic', 
                                            fill_value='extrapolate')
        estimated_price = float(interpolator(target_numeric))
    else:
        def seasonal_model(t, trend, amplitude, phase, offset):
            return trend * t + amplitude * np.sin(2 * np.pi * t / 365.25 + phase) + offset
        
        try:
            params, _ = curve_fit(seasonal_model, dates_numeric, prices, 
                                 p0=[0.001, 1, 0, 11], maxfev=10000)
            estimated_price = float(seasonal_model(target_numeric, *params))
        except:
            linear_fit = np.polyfit(dates_numeric, prices, 1)
            linear_pred = np.poly1d(linear_fit)
            target_month = target_date.month
            seasonal_adjustment = monthly_avg[target_month] - df['Prices'].mean()
            estimated_price = float(linear_pred(target_numeric) + seasonal_adjustment)
    
    return max(estimated_price, 0)



In [3]:

# ============================================================================
# 2. MAIN FUNCTION: PRICING WITH CONSTRAINTS
# ============================================================================

def calculate_contract_value_advanced(
    injection_dates,
    withdrawal_dates,
    injection_rate,
    withdrawal_rate,
    max_storage_volume,
    storage_cost_per_day
):
    """
    Calculates the value of a storage contract with multiple injections/withdrawals.
    
    Parameters :
    -----------
    injection_dates : list of str or datetime
        List of possible injection dates
        Exemple : ["2025-06-01", "2025-07-01", "2025-08-01"]
    
    withdrawal_dates : list of str or datetime
        List of possible withdrawal dates
        Exemple : ["2025-12-01", "2026-01-01", "2026-02-01"]
    
    injection_rate : float
        Maximum daily injection rate in MMBtu
        Exemple : 10_000 (can inject a maximum of 10k MMBtu/day)
    
    withdrawal_rate : float
        Maximum daily withdrawal rate in MMBtu
        Exemple : 10_000 (can withdraw a maximum of 10k MMBtu/day)
    
    max_storage_volume : float
        Maximum storage capacity in MMBtu
        Exemple : 500_000 (can store a maximum of 500k MMBtu)
    
    storage_cost_per_day : float
        Storage cost per MMBtu per day
        Exemple : 0.01 (1 cent per MMBtu per day)
    
    Return :
    --------
    dict : Detailed results including the optimal strategy and value
    """
    
    # Convert the dates to datetime if necessary
    injection_dates = [pd.to_datetime(d) if isinstance(d, str) else d 
                      for d in injection_dates]
    withdrawal_dates = [pd.to_datetime(d) if isinstance(d, str) else d 
                       for d in withdrawal_dates]
    
    # Sort the dates
    injection_dates = sorted(injection_dates)
    withdrawal_dates = sorted(withdrawal_dates)
    
    # Validation: withdrawals must be after injections
    if withdrawal_dates[0] <= injection_dates[-1]:
        print("Warning: Some withdrawal dates are before/during injections")
    
    # ==================================================================================
    # STRATEGY: GREEDY APPROACH
    # ===================================================================================
    # We will build an optimal strategy by following these rules:
    # 1. Inject on dates when the price is LOWEST
    # 2. Withdraw on dates when the price is HIGHEST
    # 3. Respect all constraints (throughput, capacity)
    
    # Calculate the price for each date
    injection_prices = [(date, estimate_price(date)) for date in injection_dates]
    withdrawal_prices = [(date, estimate_price(date)) for date in withdrawal_dates]
    
    # Sort: Injections from LOWEST to HIGHEST price
    injection_prices.sort(key=lambda x: x[1])
    
    # Sort: Prices from HIGHEST to LOWEST
    withdrawal_prices.sort(key=lambda x: x[1], reverse=True)
    
    # ========================================================================
    # STRATEGY SIMULATION
    # ========================================================================
    
    transactions = []  # List of all transactions
    current_volume = 0  # Currently stored volume
    total_injection_cost = 0
    total_withdrawal_revenue = 0
    total_storage_days = 0
    
    # Phase 1 : INJECTIONS (purchases)
    print("\n" + "="*70)
    print("INJECTION HASE (Gas Purchase)")
    print("="*70)
    
    for inj_date, inj_price in injection_prices:
        # How much can be injected? (limited by the flow rate and remaining capacity)
        max_injectable = min(
            injection_rate,  # Flow rate constraint
            max_storage_volume - current_volume  # Capacity constraint
        )
        
        if max_injectable > 0:
            # We inject the maximum possible
            volume_injected = max_injectable
            cost = volume_injected * inj_price
            
            current_volume += volume_injected
            total_injection_cost += cost
            
            transactions.append({
                'type': 'injection',
                'date': inj_date,
                'volume': volume_injected,
                'price': inj_price,
                'amount': cost,
                'storage_level': current_volume
            })
            
            print(f"{inj_date.strftime('%Y-%m-%d')} : "
                  f"Injected {volume_injected:,.0f} MMBtu @ ${inj_price:.2f}/MMBtu "
                  f"= ${cost:,.2f}")
            print(f"  Storage level : {current_volume:,.0f} / {max_storage_volume:,.0f} MMBtu")
            
            # If the storage tank is full, the injections are stopped.
            if current_volume >= max_storage_volume:
                print(f" Storage FULL - End of injections")
                break
        else:
            print(f"✗ {inj_date.strftime('%Y-%m-%d')} : "
                  f"Unable to inject (storage full)")
    
    # Calculate the average storage time
    if transactions:
        first_injection = transactions[0]['date']
        # It is assumed that withdrawal begins after the last injection
        
        # Phase 2: Withdrawals (sales)
        print("\n" + "="*70)
        print("WITHDRAWAL PHASE (Gas Sale)")
        print("="*70)
        
        # We're going to remove EVERYTHING that was injected.
        for with_date, with_price in withdrawal_prices:
            # Ensure that the withdrawal is done after the last injection
            if with_date <= transactions[-1]['date']:
                continue
            
            # How much can be withdrawn?
            max_withdrawable = min(
                withdrawal_rate,  # Constraint debit
                current_volume  # You can't take out more than you have
            )
            
            if max_withdrawable > 0:
                volume_withdrawn = max_withdrawable
                revenue = volume_withdrawn * with_price
                
                # Calculate the storage days for this volume
                days_stored = (with_date - first_injection).days
                storage_cost_this_volume = volume_withdrawn * storage_cost_per_day * days_stored
                
                current_volume -= volume_withdrawn
                total_withdrawal_revenue += revenue
                total_storage_days += (volume_withdrawn * days_stored)
                
                transactions.append({
                    'type': 'withdrawal',
                    'date': with_date,
                    'volume': volume_withdrawn,
                    'price': with_price,
                    'amount': revenue,
                    'storage_level': current_volume,
                    'storage_cost': storage_cost_this_volume
                })
                
                print(f"{with_date.strftime('%Y-%m-%d')} : "
                      f"Withdrawn {volume_withdrawn:,.0f} MMBtu @ ${with_price:.2f}/MMBtu "
                      f"= ${revenue:,.2f}")
                print(f"  Storage cost : ${storage_cost_this_volume:,.2f}")
                print(f"  Storage level : {current_volume:,.0f} / {max_storage_volume:,.0f} MMBtu")
                
                # If the storage is empty, we stop
                if current_volume <= 0:
                    print(f"   Storage EMPTY - Withdrawals complete")
                    break
            else:
                print(f" {with_date.strftime('%Y-%m-%d')} : "
                      f"Unable to remove (storage empty)")
        
        # NOTE: If there is still unsold gas, continue with the next available dates.
        if current_volume > 0:
            print(f"\n  He's staying {current_volume:,.0f} MMBtu not sold")
            print(f"    We continue with the following withdrawal dates...")
            
            # Reuse the withdrawal dates in descending price order
            for with_date, with_price in withdrawal_prices:
                if with_date <= transactions[-1]['date']:
                    continue
                    
                if current_volume > 0:
                    max_withdrawable = min(withdrawal_rate, current_volume)
                    
                    if max_withdrawable > 0:
                        volume_withdrawn = max_withdrawable
                        revenue = volume_withdrawn * with_price
                        days_stored = (with_date - first_injection).days
                        storage_cost_this_volume = volume_withdrawn * storage_cost_per_day * days_stored
                        
                        current_volume -= volume_withdrawn
                        total_withdrawal_revenue += revenue
                        
                        transactions.append({
                            'type': 'withdrawal',
                            'date': with_date,
                            'volume': volume_withdrawn,
                            'price': with_price,
                            'amount': revenue,
                            'storage_level': current_volume,
                            'storage_cost': storage_cost_this_volume
                        })
                        
                        print(f" {with_date.strftime('%Y-%m-%d')} : "
                              f"Withdrawn {volume_withdrawn:,.0f} MMBtu @ ${with_price:.2f}/MMBtu "
                              f"= ${revenue:,.2f}")
                        
                        if current_volume <= 0:
                            print(f"   All the gas has been sold !")
                            break
    
    # ========================================================================
    # NET VALUE CALCULATION
    # ========================================================================
    
    # Total storage cost
    total_storage_cost = sum(t.get('storage_cost', 0) for t in transactions)
    
    # Net worth
    net_value = total_withdrawal_revenue - total_injection_cost - total_storage_cost
    
    # Compile the results
    results = {
        'transactions': transactions,
        'total_volume_injected': sum(t['volume'] for t in transactions if t['type'] == 'injection'),
        'total_volume_withdrawn': sum(t['volume'] for t in transactions if t['type'] == 'withdrawal'),
        'total_injection_cost': total_injection_cost,
        'total_withdrawal_revenue': total_withdrawal_revenue,
        'gross_profit': total_withdrawal_revenue - total_injection_cost,
        'total_storage_cost': total_storage_cost,
        'net_value': net_value,
        'average_buy_price': total_injection_cost / sum(t['volume'] for t in transactions if t['type'] == 'injection') if any(t['type'] == 'injection' for t in transactions) else 0,
        'average_sell_price': total_withdrawal_revenue / sum(t['volume'] for t in transactions if t['type'] == 'withdrawal') if any(t['type'] == 'withdrawal' for t in transactions) else 0,
    }
    
    return results


def print_results(results):
    """Displays the results in a formatted way."""
    print("\n" + "="*70)
    print("CONTRACT SUMMARY")
    print("="*70)
    
    print(f"\n VOLUMES :")
    print(f"   • Total injected volume  : {results['total_volume_injected']:,.0f} MMBtu")
    print(f"   • Total volume removed   : {results['total_volume_withdrawn']:,.0f} MMBtu")
    
    print(f"\n AVERAGE PRICE :")
    print(f"   • Average purchase price    : ${results['average_buy_price']:.2f}/MMBtu")
    print(f"   • Average selling price   : ${results['average_sell_price']:.2f}/MMBtu")
    print(f"   • Average spread         : ${results['average_sell_price'] - results['average_buy_price']:.2f}/MMBtu")
    
    print(f"\n CASH FLOW :")
    print(f"   • Injection cost      : -${results['total_injection_cost']:,.2f}")
    print(f"   • Withdrawal income     : +${results['total_withdrawal_revenue']:,.2f}")
    print(f"   • Gross profit          : ${results['gross_profit']:,.2f}")
    print(f"   • Storage cost     : -${results['total_storage_cost']:,.2f}")
    print(f"   ─────────────────────────────")
    
    print(f"\n FINAL RESULT :")
    if results['net_value'] > 0:
        print(f"   NET VALUE        : ${results['net_value']:,.2f} (PROFITABLE)")
    elif results['net_value'] == 0:
        print(f"   NET VALUE         : ${results['net_value']:,.2f} (BREAK-EVEN)")
    else:
        print(f"   NET VALUE         : ${results['net_value']:,.2f} (PERTE)")
    
    print("="*70)



In [4]:

# ============================================================================
# 3. TESTING AND VALIDATION
# ============================================================================

if __name__ == "__main__":
    
    print("="*70)
    print("🧪 ADVANCED PRICING MODEL TESTS")
    print("="*70)
    
    # ========================================================================
    # TEST 1 : Simple Scenario (One injection, one withdrawal)
    # ========================================================================
    
    print("\n" + "─"*70)
    print("TEST 1 : Simple Scenario")
    print("─"*70)
    print("Strategy: Buy in summer, sell in winter")
    
    results1 = calculate_contract_value_advanced(
        injection_dates=["2025-08-01"],      # A single injection date
        withdrawal_dates=["2026-01-15"],     # A single withdrawal date
        injection_rate=100_000,              # Can inject 100k MMBtu/day
        withdrawal_rate=100_000,             # Can withdraw 100k MMBtu/day
        max_storage_volume=100_000,          # Capacity : 100k MMBtu
        storage_cost_per_day=0.01            # 1 cent per MMBtu per day
    )
    
    print_results(results1)
    
    # ========================================================================
    # TEST 2: Multiple Injections and Withdrawals
    # ========================================================================
    
    print("\n" + "─"*70)
    print("TEST 2 : Multiple Injections and Removals")
    print("─"*70)
    print("Strategy: Inject gradually in summer, withdraw in winter")
    
    results2 = calculate_contract_value_advanced(
        injection_dates=[
            "2025-06-15", "2025-07-15", "2025-08-15"  # 3 injection dates
        ],
        withdrawal_dates=[
            "2025-12-15", "2026-01-15", "2026-02-15"  # 3 withdrawal dates
        ],
        injection_rate=50_000,               # Limited bandwidth : 50k/day
        withdrawal_rate=50_000,
        max_storage_volume=150_000,          # Capacity : 150k MMBtu
        storage_cost_per_day=0.01
    )
    
    print_results(results2)
    
    # ========================================================================
    # TEST 3 : Active Capacity Constraint
    # ========================================================================
    
    print("\n" + "─"*70)
    print("TEST 3 : Capacity Constraint (Small Storage)")
    print("─"*70)
    print("Stratégie : Limited capacity forces optimal choices")
    
    results3 = calculate_contract_value_advanced(
        injection_dates=[
            "2025-06-01", "2025-07-01", "2025-08-01", "2025-09-01"
        ],
        withdrawal_dates=[
            "2025-11-01", "2025-12-01", "2026-01-01", "2026-02-01"
        ],
        injection_rate=30_000,
        withdrawal_rate=30_000,
        max_storage_volume=50_000,           # VERY limited capacity
        storage_cost_per_day=0.015
    )
    
    print_results(results3)
    
    # ========================================================================
    # TEST 4: Large Capacity, Numerous Dates
    # ========================================================================
    
    print("\n" + "─"*70)
    print("TEST 4 : Large Capacity with Numerous Options")
    print("─"*70)
    print("Strategy : Optimization across numerous dates")
    
    results4 = calculate_contract_value_advanced(
        injection_dates=[
            "2025-05-15", "2025-06-01", "2025-06-15",
            "2025-07-01", "2025-07-15", "2025-08-01",
            "2025-08-15", "2025-09-01"
        ],
        withdrawal_dates=[
            "2025-11-15", "2025-12-01", "2025-12-15",
            "2026-01-01", "2026-01-15", "2026-02-01",
            "2026-02-15", "2026-03-01"
        ],
        injection_rate=100_000,
        withdrawal_rate=100_000,
        max_storage_volume=500_000,          # Large capacity
        storage_cost_per_day=0.01
    )
    
    print_results(results4)
    
    print("\n" + "="*70)
    print("ALL TESTS COMPLETED")
    print("="*70)
    print("\nThe advanced pricing model is ready for production !")
    print("It manages multiple injections/withdrawals and all constraints.")
    print("="*70 + "\n")

🧪 ADVANCED PRICING MODEL TESTS

──────────────────────────────────────────────────────────────────────
TEST 1 : Simple Scenario
──────────────────────────────────────────────────────────────────────
Strategy: Buy in summer, sell in winter

INJECTION HASE (Gas Purchase)
2025-08-01 : Injected 100,000 MMBtu @ $12.05/MMBtu = $1,204,673.71
  Storage level : 100,000 / 100,000 MMBtu
 Storage FULL - End of injections

WITHDRAWAL PHASE (Gas Sale)
2026-01-15 : Withdrawn 100,000 MMBtu @ $13.64/MMBtu = $1,364,453.08
  Storage cost : $167,000.00
  Storage level : 0 / 100,000 MMBtu
   Storage EMPTY - Withdrawals complete

CONTRACT SUMMARY

 VOLUMES :
   • Total injected volume  : 100,000 MMBtu
   • Total volume removed   : 100,000 MMBtu

 AVERAGE PRICE :
   • Average purchase price    : $12.05/MMBtu
   • Average selling price   : $13.64/MMBtu
   • Average spread         : $1.60/MMBtu

 CASH FLOW :
   • Injection cost      : -$1,204,673.71
   • Withdrawal income     : +$1,364,453.08
   • Gross profit